# Generating Icepack Inputs for Runs

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import csv
import sys
import itertools

### Config

In [2]:
parameter_dict = {"ksno":[.14,.43]}
parameterization_dict = {"ktherm":[2]}
locs = ["centralarctic", "barents", "sibchuk", "coastalcanada"]
var_num = 8
pdir = "MDF_1"
SEED = 47

In [11]:
OUT_DIR = Path("/projects/dagr5998/icepack_ML/parameters") / pdir
OUT_DIR.mkdir(parents=True)

FileExistsError: [Errno 17] File exists: '/projects/dagr5998/icepack_ML/parameters/MDF_1'

## Generating Record and Values

In [5]:
def sample_cont(param_dict, n):
    sample_dict = {}
    for key, hi_lo in param_dict.items():
        sample_dict[key] = np.round(np.linspace(hi_lo[0], hi_lo[1], n),4) # includes the endpoints, won't matter when replaced with random
    return sample_dict

In [26]:
def to_fortran(x):
    if type(x) is bool:
        return ".true." if x else ".false."
    if type(x) is int or type(x) is np.int32 or type(x) is np.int64:
        return str(x)
    if type(x) is str:
        return f"'{x}'"
    return f"{x:.10e}".replace("e", "d")

In [13]:
cont_pts = sample_cont(parameter_dict, var_num)
dis_pts = parameterization_dict
tot_pts = cont_pts | dis_pts

In [14]:
combinations = [dict(zip(tot_pts.keys(), combo)) for combo in itertools.product(*tot_pts.values())]

In [32]:
ind=0
for run_i in combinations:
    ind +=1
    set_id = f"{ind:04d}"
    lines = []
    for key, val in run_i.items():
       lines += [f"{key} = {to_fortran(val)}\n"]
    
    mod_path = OUT_DIR / f"set_{set_id}.mods"
    with open(mod_path, "w", encoding="utf-8") as file:
        file.writelines(lines)